Worked on by: Thomas Van Sande

In this notebook we'll build theforecasting model for the demand dataset using PyCaret.

# Predict

In [35]:
%pip uninstall dask -y
%pip uninstall distributed -y
%pip install "dask[dataframe]==2023.9.2" distributed==2023.9.2

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


     ---------------------------------------- 1.2/1.2 MB 7.6 MB/s eta 0:00:00
     ------------------------------------- 994.9/994.9 kB 12.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
%pip install pandas matplotlib numpy
%pip install openpyxl
%pip install pycaret

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import pickle
with open("cleaned_data.pkl", "rb") as file:
    df = pickle.load(file)


In [38]:
df.to_csv("cleaned_data.csv", index=False)

In [39]:
df.head()

,settlement_date,settlement_period,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
0,2006-01-01,1,38596,39660.0,34982,0.0,0.0,0.0,0.0,0,295,1997,0.0,-169.0
1,2006-01-01,2,38829,39897.0,35312,0.0,0.0,0.0,0.0,0,299,1997,0.0,-169.0
2,2006-01-01,3,38456,39599.0,35018,0.0,0.0,0.0,0.0,0,374,1998,0.0,-169.0
3,2006-01-01,4,37401,38823.0,34054,0.0,0.0,0.0,0.0,0,653,1998,0.0,-169.0
4,2006-01-01,5,36586,37937.0,33297,0.0,0.0,0.0,0.0,0,582,1998,0.0,-169.0


In [40]:
import pandas as pd
df['year'] = pd.DatetimeIndex(df['settlement_date']).year
df['month'] = pd.DatetimeIndex(df['settlement_date']).month
df['day'] = pd.DatetimeIndex(df['settlement_date']).day
df = df.drop(columns=['settlement_date'])

I want the granularity to be per month, since per day seems quite small for forecasting electricity demand

In [41]:
df_monthly = df.drop(columns=["day"]).groupby(["year","month"], as_index=False).mean()


In [42]:
df_monthly.head()

,year,month,settlement_period,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
0,2006,1,24.500000,44926.580645,46292.341398,40684.339382,0.0,0.0,0.0,0.0,2.492608,486.573925,1163.925403,0.0,-175.514785
1,2006,2,24.500000,45075.880208,46519.120536,40857.686756,0.0,0.0,0.0,0.0,2.793899,463.634673,597.453869,0.0,-93.101190
2,2006,3,24.469044,44233.642665,45466.306864,40027.205249,0.0,0.0,0.0,0.0,1.619112,467.188425,1425.873486,0.0,-124.381561
3,2006,4,24.500000,37960.129861,38886.315972,34293.731250,0.0,0.0,0.0,0.0,0.002083,272.236111,1901.654167,0.0,-153.297222
4,2006,5,24.500000,35744.225134,36721.996640,32444.974462,0.0,0.0,0.0,0.0,1.083333,347.650538,1739.044355,0.0,-124.606183


In [43]:
df_monthly.tail()

,year,month,settlement_period,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
233,2025,6,24.5,21847.457639,24444.728472,20143.653472,2002.455556,6606.0,3600.309028,20784.559028,0.0,129.782639,1004.897222,29.010417,-269.759722
234,2025,7,24.5,22665.625000,24896.304435,20755.875672,1188.024194,6606.0,3163.289651,20894.651882,0.0,93.918683,1539.971774,-43.504032,-175.690188
235,2025,8,24.5,21808.459005,24206.637769,19987.661290,1581.700269,6606.0,2909.141129,20396.096774,0.0,85.360215,1690.110215,48.041667,-277.826613
236,2025,9,24.5,23629.193750,26154.865278,21733.675694,2037.978472,6606.0,2292.840278,20993.000000,0.0,183.377778,1025.290278,-92.545833,-274.762500
237,2025,10,24.5,25664.076389,27957.263889,23998.680556,3111.798611,6606.0,947.750000,20993.000000,0.0,150.006944,516.368056,-257.326389,-85.472222


In [44]:
df_monthly = df_monthly.drop('settlement_period', axis=1)

In [45]:
df_monthly.head()

,year,month,nd,tsd,england_wales_demand,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,britned_flow,moyle_flow
0,2006,1,44926.580645,46292.341398,40684.339382,0.0,0.0,0.0,0.0,2.492608,486.573925,1163.925403,0.0,-175.514785
1,2006,2,45075.880208,46519.120536,40857.686756,0.0,0.0,0.0,0.0,2.793899,463.634673,597.453869,0.0,-93.101190
2,2006,3,44233.642665,45466.306864,40027.205249,0.0,0.0,0.0,0.0,1.619112,467.188425,1425.873486,0.0,-124.381561
3,2006,4,37960.129861,38886.315972,34293.731250,0.0,0.0,0.0,0.0,0.002083,272.236111,1901.654167,0.0,-153.297222
4,2006,5,35744.225134,36721.996640,32444.974462,0.0,0.0,0.0,0.0,1.083333,347.650538,1739.044355,0.0,-124.606183


Now we use PyCaret for building the model using the monthly granularity. We'll forecast the next 6 months

In [46]:
from pycaret.time_series import *
s = setup(
    data=df_monthly,
    target="england_wales_demand",
    fh=6,          
    session_id=123,
    fold=5
)

AttributeError: module 'dask.dataframe.core' has no attribute 'DataFrame'

In [ ]:
best = compare_models()
# Added by Iljas, saving a different way
final_best = finalize_model(best)
save_model(final_best, "different_save_best_model")

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
auto_arima,Auto ARIMA,0.0686,0.0613,79.9357,93.5907,0.0033,0.0033,0.9971,53.3160
arima,ARIMA,0.0699,0.0669,81.4549,102.2543,0.0033,0.0033,0.9965,1.7240
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,0.5155,0.4732,598.3966,721.3590,0.0239,0.0241,0.8529,0.1580
lightgbm_cds_dt,Light Gradient Boosting w/ Cond. Deseasonalize & Detrending,0.5264,0.4908,611.8362,748.8297,0.0244,0.0248,0.8316,0.2280
en_cds_dt,Elastic Net w/ Cond. Deseasonalize & Detrending,0.5703,0.5308,663.8118,810.8851,0.0270,0.0272,0.7955,0.3820
ridge_cds_dt,Ridge w/ Cond. Deseasonalize & Detrending,0.5841,0.5425,680.5376,829.6575,0.0276,0.0276,0.7794,0.3760
llar_cds_dt,Lasso Least Angular Regressor w/ Cond. Deseasonalize & Detrending,0.5843,0.5427,680.7074,829.9558,0.0276,0.0276,0.7792,0.1360
lasso_cds_dt,Lasso w/ Cond. Deseasonalize & Detrending,0.5844,0.5428,680.8601,830.1976,0.0276,0.0276,0.7792,0.2920
lr_cds_dt,Linear w/ Cond. Deseasonalize & Detrending,0.5852,0.5432,681.8757,830.7970,0.0276,0.0276,0.7785,1.3820
br_cds_dt,Bayesian Ridge w/ Cond. Deseasonalize & Detrending,0.5963,0.5469,693.1112,834.5232,0.0281,0.0285,0.7731,0.1340


Auto arima provided the best model. Not surprising since auto arima is very good at dealing with seasonality and this dataset has very clear seasonality. We now save this model as a pickle file so we don't have to build this again and again when we want to use this. The evaluation metrics seem very good. An RMSE of 93 when dealing with numbers over 40k seems very accurate already. We'll also visualize this next.

In [ ]:
with open('best_model.pkl', 'wb') as file:
    pickle.dump(best, file)

In [ ]:
plot_model(best, plot = 'forecast', data_kwargs = {'fh' : 6})

As you can see, the model is very close to the actual data of the last 6 months

In [ ]:
import pickle

with open("best_model.pkl", "rb") as file:
    best = pickle.load(file)


In [ ]:
best.get_params()

{'D': None,
 'alpha': 0.05,
 'concentrate_scale': False,
 'd': None,
 'enforce_invertibility': True,
 'enforce_stationarity': True,
 'error_action': 'warn',
 'hamilton_representation': False,
 'information_criterion': 'aic',
 'max_D': 1,
 'max_P': 2,
 'max_Q': 2,
 'max_d': 2,
 'max_order': 5,
 'max_p': 5,
 'max_q': 5,
 'maxiter': 50,
 'measurement_error': False,
 'method': 'lbfgs',
 'mle_regression': True,
 'n_fits': 10,
 'n_jobs': 1,
 'offset_test_args': None,
 'out_of_sample_size': 0,
 'random': False,
 'random_state': 123,
 'scoring': 'mse',
 'scoring_args': None,
 'seasonal': True,
 'seasonal_test': 'ocsb',
 'seasonal_test_args': None,
 'simple_differencing': False,
 'sp': 12,
 'start_P': 1,
 'start_Q': 1,
 'start_p': 2,
 'start_params': None,
 'start_q': 2,
 'stationary': False,
 'stepwise': True,
 'suppress_warnings': True,
 'test': 'kpss',
 'time_varying_regression': False,
 'trace': False,
 'trend': None,
 'update_pdq': True,
 'with_intercept': True}

In [ ]:
best.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                      
==========================================================================================
Dep. Variable:                                  y   No. Observations:                  232
Model:             SARIMAX(2, 0, 0)x(2, 0, 0, 12)   Log Likelihood               -1336.737
Date:                            Mon, 24 Nov 2025   AIC                           2709.474
Time:                                    11:05:30   BIC                           2771.516
Sample:                                         0   HQIC                          2734.495
                                            - 232                                         
Covariance Type:                              opg                                         
=============================================================================================
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
year                          0.2928      0.092      3.199      0.001       0.113       0.472
month                        -6.7871      3.064     -2.215      0.027     -12.791      -0.783
nd                            0.8563      0.024     36.183      0.000       0.810       0.903
tsd                           0.0330      0.023      1.421      0.155      -0.012       0.078
embedded_wind_generation      0.1509      0.019      7.749      0.000       0.113       0.189
embedded_wind_capacity       -0.0639      0.023     -2.767      0.006      -0.109      -0.019
embedded_solar_generation    -0.1192      0.028     -4.258      0.000      -0.174      -0.064
embedded_solar_capacity       0.0271      0.008      3.432      0.001       0.012       0.043
non_bm_stor                   0.7338      1.263      0.581      0.561      -1.741       3.209
pump_storage_pumping          0.1214      0.155      0.784      0.433      -0.182       0.425
ifa_flow                     -0.0260      0.015     -1.709      0.087      -0.056       0.004
britned_flow                  0.0054      0.037      0.146      0.884      -0.066       0.077
moyle_flow                    0.0040      0.086      0.047      0.963      -0.164       0.172
ar.L1                         0.3381      0.069      4.899      0.000       0.203       0.473
ar.L2                         0.1984      0.070      2.829      0.005       0.061       0.336
ar.S.L12                      0.2658      0.076      3.478      0.001       0.116       0.416
ar.S.L24                      0.1564      0.080      1.955      0.051      -0.000       0.313
sigma2                     6551.6680    728.541      8.993      0.000    5123.753    7979.583
===================================================================================
Ljung-Box (L1) (Q):                   0.17   Jarque-Bera (JB):                 0.91
Prob(Q):                              0.68   Prob(JB):                         0.64
Heteroskedasticity (H):               1.82   Skew:                             0.15
Prob(H) (two-sided):                  0.01   Kurtosis:                         3.07
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""